# Invoice Processing Agent — Colab run

Run the cells top to bottom. Cell 6 is the diagnostic — that's the one to screenshot.

You do **not** need an API key. Everything runs offline in deterministic mode.
Cell 8 is optional and adds real LLM reasoning.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Unzip the project

**Use the zip, not the loose files.** The loose files are flat; the project needs
`agents/` and `db/` subfolders plus the 20 invoices, and the zip has all of that.

Edit `ZIP_PATH` below to point at your `invoice-agent.zip`.

In [ ]:
import zipfile, os, shutil

ZIP_PATH = '/content/drive/MyDrive/files-3/invoice-agent.zip'   # <-- EDIT THIS

assert os.path.exists(ZIP_PATH), f'Not found: {ZIP_PATH}\nCheck the path in the file browser on the left.'

shutil.rmtree('/content/invoice-agent', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/')

os.chdir('/content/invoice-agent')
print('Working in:', os.getcwd())
print('\nStructure:')
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if d not in ('__pycache__', '.git')]
    depth = root.count(os.sep)
    if depth > 2: continue
    print('  ' * depth + os.path.basename(root) + '/')
    if 'invoices' not in root:
        for f in sorted(files)[:12]:
            print('  ' * (depth + 1) + f)
    else:
        print('  ' * (depth + 1) + f'({len(files)} invoice files)')

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import importlib
for mod, label in [('langgraph', 'LangGraph'), ('langchain_core', 'LangChain Core'),
                   ('pydantic', 'Pydantic'), ('pdfplumber', 'pdfplumber'),
                   ('groq', 'Groq SDK'), ('dotenv', 'python-dotenv')]:
    try:
        importlib.import_module(mod)
        print(f'  ok   {label}')
    except ImportError as e:
        print(f'  MISSING  {label}  ({e})')

## 4. Single invoice — the command the brief asks for

**Screenshot this.** It proves the required interface works.

In [ ]:
!python main.py --invoice_path=data/invoices/invoice_1002.txt

## 5. The invoice that shows off the catching

INV-1013 hides a **$50 error inside a $22,562 total**, and requests 22 WidgetA
spread across three lines of 15, 5 and 2 — each line passes stock on its own.

**Screenshot this too.**

In [ ]:
!python main.py --invoice_path=data/invoices/invoice_1013.json

## 6. Diagnostic — the important one

84 checks: every agent on real data, plus guardrails attacked deliberately
(corrupt files, a hostile LLM that always approves, forced payments, repeat runs).

**Screenshot the summary at the bottom.**

In [ ]:
!python diagnostic.py

## 7. Full batch + dashboard

Processes all 20 files and writes the review queue.

In [ ]:
!python run_batch.py

In [ ]:
# Render the dashboard inline. Screenshot the top section.
from IPython.display import HTML, display
display(HTML(open('logs/invoice-review.html').read()))

## 8. Optional — one run with a real LLM

Free key from **console.groq.com**. This is the only way to prove the AI parts
and the critique loop actually fire, which is a scored criterion.

Look for `[approve_draft] LLM decision=` and `[approve_critique]` in the output.

In [ ]:
import os
os.environ['GROQ_API_KEY'] = ''        # <-- paste your gsk_... key here

if os.environ.get('GROQ_API_KEY'):
    !python main.py --invoice_path=data/invoices/invoice_1012.txt
else:
    print('No key set — skipping. The offline runs above are already complete and valid.')

In [ ]:
# Clear the key so committed logs are the offline ones
os.environ.pop('GROQ_API_KEY', None)
print('Key cleared.')

## 9. Package for GitHub

Downloads a clean zip with no database, no cache, no keys.

In [ ]:
import shutil, os, glob

for pattern in ['inventory.db', 'logs/run_*.json', 'logs/impact.json', '.env']:
    for f in glob.glob(pattern):
        os.remove(f)
for root, dirs, _ in os.walk('.'):
    for d in list(dirs):
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d), ignore_errors=True)

os.chdir('/content')
shutil.make_archive('/content/invoice-agent-final', 'zip', '/content', 'invoice-agent')
os.chdir('/content/invoice-agent')

from google.colab import files
files.download('/content/invoice-agent-final.zip')

## 10. Push to GitHub

Unzip the download locally, then:

```bash
cd invoice-agent
git init
git add .
git commit -m "Invoice processing agent: LangGraph multi-agent pipeline"
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPO.git
git push -u origin main
```

`.gitignore` already excludes the key, the database, and run logs.
Commit `logs/invoice-review.html` so reviewers see the output without running anything.